# Principal Portfolios with Dynamic Risk Overlays

Dynamic, implementation-aware risk overlays for investable PCA factor portfolios.

**Core scope.** This notebook focuses on the overlay layer only:
- walk-forward investable PC1 as the base portfolio;
- probabilistic volatility-state overlays;
- Corwin--Schultz spread-based implementation costs;
- QQQ / equal-weight benchmarks;
- stationary block-bootstrap OOS inference.

Regime-conditioned performance analysis and PC-rotation diagnostics are intentionally excluded here; they belong in the upstream PCA/regime project.


In [ ]:
# ============================================================
# 0) Imports and configuration
# ============================================================

from yfinance import download
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.stats import norm
from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 4)
pd.options.display.float_format = "{:,.6f}".format

# ------------------------------------------------------------
# Universe and market proxies
# ------------------------------------------------------------
TICKERS_US_TECH = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "QCOM", "CSCO", "NVDA",
    "ORCL", "TXN", "ADBE", "IBM", "CRM", "AMAT", "INTU"
]

IV_TICKER_PRIMARY = "^VXN"   # Nasdaq-100 implied volatility
IV_TICKER_FALLBACK = "^VIX"  # fallback if ^VXN is unavailable
RF_TICKER_PRIMARY = "^IRX"   # 13-week Treasury bill annualized yield proxy

START = "2012-01-01"
END   = "2026-03-31"

# Development / OOS split for overlays.
TRAIN_END_DEFAULT = "2017-12-31"
TEST_START_DEFAULT = "2018-01-01"

# PCA / walk-forward settings.
K_BAR = 7
GROSS_NORM = 1.0
WF_TRAIN_YEARS = 5
WF_STEP_MONTHS = 3
WF_K_CHOICE = 7
PC_CORE = "PC1"

ANN = 252
RF_ANN = 252

# Bootstrap settings.
BOOTSTRAP_BLOCK_LEN = 90   # overwritten by ACF matching if calibration succeeds
BOOTSTRAP_BLOCK_METHOD = "fallback fixed value"


In [ ]:
# ============================================================
# 1) Reusable helpers
# ============================================================

def linear_returns_from_prices(adj_close: pd.DataFrame | pd.Series) -> pd.DataFrame | pd.Series:
    return adj_close.pct_change().dropna(how="all")


def annualized_yield_pct_to_daily_return(y_ann_pct: pd.Series, ann: int = 252) -> pd.Series:
    y = pd.Series(y_ann_pct).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    y = (y / 100.0).clip(lower=-0.999999)
    out = (1.0 + y) ** (1.0 / ann) - 1.0
    out.name = "rf_daily"
    return out


def weighted_mean_cov(X: pd.DataFrame):
    mu = X.mean(axis=0).values
    Xc = X.values - mu.reshape(1, -1)
    Sigma = (Xc.T @ Xc) / X.shape[0]
    return (
        pd.Series(mu, index=X.columns, name="mu"),
        pd.DataFrame(Sigma, index=X.columns, columns=X.columns),
    )


def correlation_pca(Sigma: np.ndarray):
    Sigma = np.asarray(Sigma, dtype=float)
    vol = np.sqrt(np.clip(np.diag(Sigma), 1e-18, None))
    Dinv = np.diag(1.0 / vol)
    rho = Dinv @ Sigma @ Dinv
    lam, V = np.linalg.eigh(rho)
    lam = lam[::-1]
    V = V[:, ::-1]

    # Deterministic sign convention.
    max_abs_row = np.argmax(np.abs(V), axis=0)
    s = np.sign(V[max_abs_row, np.arange(V.shape[0])])
    s[s == 0] = 1.0
    V = V * s.reshape(1, -1)
    return lam, V, vol, rho


def pc_portfolio_weights_from_corr_pca(V: np.ndarray, vol: np.ndarray, gross_norm: float = 1.0) -> pd.DataFrame:
    V = np.asarray(V, dtype=float)
    vol = np.asarray(vol, dtype=float)
    W = V * (1.0 / vol).reshape(-1, 1)
    gross = np.sum(np.abs(W), axis=0)
    gross[gross == 0] = 1.0
    W = W / gross.reshape(1, -1) * gross_norm
    return pd.DataFrame(W)


def factor_returns(R: pd.DataFrame, W: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        R.values @ W.values,
        index=R.index,
        columns=[f"PC{i+1}" for i in range(W.shape[1])],
    )


def max_drawdown(r: pd.Series) -> float:
    r = pd.Series(r).dropna().astype(float)
    if r.empty:
        return np.nan
    w = (1.0 + r).cumprod()
    dd = w / w.cummax() - 1.0
    return float(dd.min())


def perf_table(return_dict: dict[str, pd.Series], ann: int = 252) -> pd.DataFrame:
    rows = []
    for name, r in return_dict.items():
        r = pd.Series(r).dropna().astype(float)
        if len(r) < 5:
            rows.append({"strategy": name, "n": len(r)})
            continue

        w = (1.0 + r).cumprod()
        n = len(r)
        yrs = n / ann
        cagr = w.iloc[-1] ** (1.0 / yrs) - 1.0 if yrs > 0 else np.nan
        ann_ret = r.mean() * ann
        ann_vol = r.std(ddof=1) * np.sqrt(ann)
        sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan

        downside = r[r < 0]
        down_vol = downside.std(ddof=1) * np.sqrt(ann) if len(downside) > 1 else np.nan
        sortino = ann_ret / down_vol if pd.notna(down_vol) and down_vol > 0 else np.nan

        peak = w.cummax()
        dd = w / peak - 1.0
        mdd = dd.min()
        calmar = cagr / abs(mdd) if mdd < 0 else np.nan
        ulcer = np.sqrt(np.mean((100.0 * dd) ** 2)) / 100.0
        martin = cagr / ulcer if ulcer > 0 else np.nan

        rows.append({
            "strategy": name,
            "n": n,
            "CAGR": cagr,
            "AnnVol": ann_vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "MaxDD": mdd,
            "Calmar": calmar,
            "Martin": martin,
        })
    return pd.DataFrame(rows).set_index("strategy")


def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))


def rolling_realized_vol(r: pd.Series, window: int = 21, ann: int = 252, min_obs: int | None = None) -> pd.Series:
    r = pd.Series(r).astype(float)
    if min_obs is None:
        min_obs = max(5, window // 2)
    return (r.rolling(window, min_periods=min_obs).std(ddof=1) * np.sqrt(ann)).rename(f"rv_{window}")


def rolling_downside_semivol(r: pd.Series, window: int = 21, ann: int = 252, mar: float = 0.0) -> pd.Series:
    r = pd.Series(r).astype(float)
    neg = np.minimum(r - mar, 0.0)
    return np.sqrt(
        ann * pd.Series(neg**2, index=r.index).rolling(window, min_periods=max(5, window // 2)).mean()
    ).rename(f"dsv_{window}")


def forward_realized_vol_from_returns(r: pd.Series, horizon: int = 21, ann: int = 252) -> pd.Series:
    r = pd.Series(r).dropna().astype(float)
    x = r.values
    out = np.full(len(x), np.nan)
    for i in range(len(x)):
        j = min(len(x), i + horizon + 1)
        fwd = x[i + 1:j]
        if len(fwd) < max(5, horizon // 2):
            continue
        out[i] = np.std(fwd, ddof=1) * np.sqrt(ann)
    return pd.Series(out, index=r.index, name=f"rv_fwd_{horizon}")


def stationary_bootstrap_indices(T: int, avg_block_len: float, rng: np.random.Generator) -> np.ndarray:
    p = 1.0 / avg_block_len
    idx = np.empty(T, dtype=int)
    idx[0] = rng.integers(0, T)
    for t in range(1, T):
        if rng.random() < p:
            idx[t] = rng.integers(0, T)
        else:
            idx[t] = (idx[t - 1] + 1) % T
    return idx


def _acf(x, nlags=30):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) < nlags + 5:
        return np.full(nlags + 1, np.nan)
    x = x - x.mean()
    denom = np.dot(x, x)
    if denom <= 0:
        return np.full(nlags + 1, np.nan)
    out = [1.0]
    for k in range(1, nlags + 1):
        out.append(float(np.dot(x[:-k], x[k:]) / denom))
    return np.asarray(out)


def choose_block_length_by_acf_matching(
    x,
    candidates,
    nlags=30,
    n_boot=300,
    seed=1,
    stationary_bootstrap_indices_fn=stationary_bootstrap_indices,
    distance="weighted",
):
    x = pd.Series(x).dropna().astype(float)
    target = _acf(x, nlags=nlags)
    rng = np.random.default_rng(seed)
    rows = []
    for L in candidates:
        sims = []
        for _ in range(n_boot):
            idx = stationary_bootstrap_indices_fn(len(x), float(L), rng)
            sims.append(_acf(x.iloc[idx].reset_index(drop=True), nlags=nlags))
        sim_mean = np.nanmean(np.vstack(sims), axis=0)
        diff = sim_mean[1:] - target[1:]
        if distance == "weighted":
            weights = 1.0 / np.arange(1, nlags + 1)
            dist = float(np.nansum(weights * diff**2))
        else:
            dist = float(np.nanmean(diff**2))
        rows.append({"block_len": L, "acf_distance": dist})
    scores = pd.DataFrame(rows).sort_values("acf_distance")
    return {"best_L": int(scores.iloc[0]["block_len"]), "scores": scores}


def politis_white_block_length(x, use_abs=True):
    """
    Lightweight diagnostic inspired by Politis--White logic.
    Used as a secondary reference only; the notebook selects block length by ACF matching.
    """
    s = pd.Series(x).dropna().astype(float)
    if use_abs:
        s = s.abs()
    ac = _acf(s, nlags=min(60, max(5, len(s)//10)))
    # crude persistence-to-length heuristic
    pos = np.where(ac[1:] <= 0)[0]
    first_zero = int(pos[0] + 1) if len(pos) else min(43, len(ac) - 1)
    return max(5, min(90, first_zero * 3))


def add_months(ts: pd.Timestamp, m: int) -> pd.Timestamp:
    return (pd.Timestamp(ts) + pd.offsets.DateOffset(months=m)).normalize()


def _align_W_to_prev(W_prev: pd.DataFrame | None, W_now: pd.DataFrame) -> pd.DataFrame:
    if W_prev is None:
        return W_now.copy()
    W_aligned = W_now.copy()
    common_cols = [c for c in W_now.columns if c in W_prev.columns]
    for c in common_cols:
        if np.dot(W_prev[c].values.astype(float), W_now[c].values.astype(float)) < 0:
            W_aligned[c] = -W_aligned[c]
    return W_aligned


def walk_forward_pca(
    R: pd.DataFrame,
    start_test: str,
    train_years: int = 5,
    step_months: int = 3,
    k: int = 7,
    gross_norm: float = 1.0,
    asset_costs: pd.DataFrame | None = None,
    charge_initial_rebalance: bool = False,
):
    R = R.dropna().sort_index().copy()
    t0 = pd.Timestamp(start_test)
    t_end = R.index.max()

    if asset_costs is not None:
        asset_costs = (
            asset_costs.reindex(index=R.index, columns=R.columns)
            .sort_index()
            .ffill()
            .fillna(0.0)
            .astype(float)
        )

    weights_hist, f_all, costs_all = [], [], []
    W_prev = None
    reb = t0

    while True:
        train_start = reb - pd.DateOffset(years=train_years)
        train_end = reb - pd.DateOffset(days=1)
        test_start = reb
        test_end = add_months(reb, step_months) - pd.DateOffset(days=1)

        if test_start > t_end:
            break

        R_train = R.loc[train_start:train_end]
        R_test = R.loc[test_start:test_end]

        if len(R_train) < 252 or len(R_test) < 5:
            reb = add_months(reb, step_months)
            continue

        _, Sigma = weighted_mean_cov(R_train)
        _, V, vol, _ = correlation_pca(Sigma.values)
        W = pc_portfolio_weights_from_corr_pca(V[:, :k], vol, gross_norm=gross_norm)
        W.index = R.columns
        W.columns = [f"PC{i+1}" for i in range(W.shape[1])]
        W = _align_W_to_prev(W_prev, W)

        f = factor_returns(R_test, W)
        f_all.append(f)

        seg_cost = pd.DataFrame(0.0, index=R_test.index, columns=f.columns)
        if asset_costs is not None and (charge_initial_rebalance or W_prev is not None):
            dW = W.abs() if W_prev is None else (W - W_prev).abs()
            trade_date = R_test.index[0]
            c_assets = asset_costs.loc[trade_date, R.columns]
            c_pc = dW.mul(c_assets, axis=0).sum(axis=0)
            seg_cost.iloc[0, :] = c_pc.reindex(f.columns).values
        costs_all.append(seg_cost)

        W_prev = W.copy()
        weights_hist.append({
            "rebalance": reb,
            "train_start": train_start,
            "train_end": train_end,
            "W": W.copy(),
        })
        reb = add_months(reb, step_months)

    F = pd.concat(f_all).sort_index() if f_all else pd.DataFrame()
    C = pd.concat(costs_all).sort_index() if costs_all else pd.DataFrame(0.0, index=F.index, columns=F.columns)
    C = C.reindex(index=F.index, columns=F.columns).fillna(0.0)
    return F, F - C, weights_hist


def factor_weights_daily_from_history(
    weights_hist,
    factor_name: str,
    target_index: pd.DatetimeIndex,
    tickers: list[str],
) -> pd.DataFrame:
    rows = []
    for item in weights_hist:
        if not isinstance(item, dict):
            continue
        reb = pd.Timestamp(item["rebalance"])
        W = item["W"].copy()
        if factor_name not in W.columns:
            continue
        s = W[factor_name].reindex(tickers).astype(float)
        s.name = reb
        rows.append(s)
    if not rows:
        raise ValueError(f"No weights found for factor {factor_name}.")
    W_step = pd.DataFrame(rows)
    W_step.index = pd.to_datetime(W_step.index)
    W_step = W_step.sort_index()
    return W_step.reindex(target_index, method="ffill")


## 2) Data download and base panels

Downloads adjusted close, high/low prices for Corwin--Schultz cost estimates, the implied-volatility proxy, QQQ benchmark, and risk-free proxy.


In [ ]:
# ============================================================
# 2) Data download
# ============================================================

all_tickers = TICKERS_US_TECH + [IV_TICKER_PRIMARY, IV_TICKER_FALLBACK]
px = download(all_tickers, start=START, end=END, auto_adjust=False, progress=False).dropna(how="all")

px_adj_close_all = px["Adj Close"][TICKERS_US_TECH].copy()
px_high = px["High"][TICKERS_US_TECH].copy()
px_low = px["Low"][TICKERS_US_TECH].copy()

# Common-support equity panel for investable PCA portfolios.
px_stocks = px_adj_close_all.dropna(how="any")
R = linear_returns_from_prices(px_stocks)

# Implied volatility proxy for fast layer.
px_adj = px["Adj Close"].copy()
iv = px_adj[IV_TICKER_PRIMARY] if IV_TICKER_PRIMARY in px_adj.columns else None
if iv is None or iv.dropna().empty:
    iv = px_adj[IV_TICKER_FALLBACK]
iv.name = "IV"

# QQQ benchmark.
px_QQQ = download("QQQ", start=START, end=END, auto_adjust=False, progress=False)["Adj Close"].dropna(how="all")
B_ret = linear_returns_from_prices(px_QQQ)
if isinstance(B_ret, pd.DataFrame):
    B_ret = B_ret.iloc[:, 0]
B_ret = pd.Series(B_ret).dropna().astype(float)
B_ret.name = "QQQ"

# Historical risk-free proxy.
rf_raw = download(RF_TICKER_PRIMARY, start=START, end=END, auto_adjust=False, progress=False)

def _extract_single_close(x):
    if isinstance(x, pd.Series):
        return x.astype(float)
    if not isinstance(x, pd.DataFrame):
        return pd.Series(dtype=float)
    for field in ["Adj Close", "Close"]:
        if field in x.columns:
            s = x[field]
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            return s.astype(float)
    if x.shape[1] == 1:
        return x.iloc[:, 0].astype(float)
    return pd.Series(dtype=float)

rf_ann_pct = _extract_single_close(rf_raw).dropna()
if rf_ann_pct.empty:
    print(f"WARNING: no historical risk-free proxy downloaded from {RF_TICKER_PRIMARY}; using zero daily RF.")
    rf_daily_hist = pd.Series(0.0, index=R.index, name="rf_daily")
    rf_source_used = "ZERO_FALLBACK"
else:
    rf_daily_hist = annualized_yield_pct_to_daily_return(rf_ann_pct, ann=RF_ANN)
    rf_daily_hist = rf_daily_hist.reindex(R.index).ffill().fillna(0.0)
    rf_source_used = RF_TICKER_PRIMARY

print("Equity panel:", R.index.min().date(), "->", R.index.max().date(), "| shape:", R.shape)
print("IV proxy:", iv.name, "| non-null obs:", int(iv.notna().sum()))
print("QQQ benchmark:", B_ret.index.min().date(), "->", B_ret.index.max().date(), "| n:", len(B_ret))
print("RF source:", rf_source_used)
display(R.head())


## 3) Bootstrap block-length calibration

Select the mean block length by matching the ACF of absolute QQQ returns. This is used for downstream stationary block-bootstrap OOS inference.

In [ ]:
# ============================================================
# Bootstrap block-length calibration (QQQ benchmark)
# ============================================================

px_B = download("QQQ", start=START, end=END, auto_adjust=False, progress=False)["Adj Close"].dropna(how="all")
B_ret = linear_returns_from_prices(px_B)

if isinstance(B_ret, pd.DataFrame):
    B_ret = B_ret.iloc[:, 0]
elif isinstance(B_ret, np.ndarray) and B_ret.ndim == 2 and B_ret.shape[1] == 1:
    B_ret = B_ret[:, 0]
B_ret = pd.Series(B_ret).dropna().astype(float)
B_ret.name = "QQQ"

# We use ACF-matching on |QQQ returns| as the primary selector because the
# combined-overlay bootstrap is especially sensitive to volatility clustering
# and drawdown-path dependence.  The Politis--White-style value is kept as a
# diagnostic, but the implementation below is a lightweight heuristic, not the
# exact original estimator.
BLOCK_CANDIDATES = [5, 10, 15, 20, 30, 43, 60, 90]
res_abs = choose_block_length_by_acf_matching(
    B_ret.abs(),
    candidates=BLOCK_CANDIDATES,
    nlags=30,
    n_boot=300,
    use_abs=True,
    seed=1,
    stationary_bootstrap_indices_fn=stationary_bootstrap_indices,
    distance="weighted",
)

BOOTSTRAP_BLOCK_LEN = int(res_abs["best_L"])
BOOTSTRAP_BLOCK_METHOD = "ACF-matching on |QQQ returns|"

print("ACF-matching best L on |QQQ returns|:", BOOTSTRAP_BLOCK_LEN)
display(res_abs["scores"])

L_pw = politis_white_block_length(B_ret, use_abs=True)
print("Politis--White-style suggested mean block length:", L_pw)
print(f"Selected bootstrap mean block length for downstream validation: {BOOTSTRAP_BLOCK_LEN} ({BOOTSTRAP_BLOCK_METHOD})")



In [ ]:
# ============================================================
# Corwin–Schultz helpers (single asset + panel + one-way cost)
# ============================================================

def corwin_schultz_spread(high: pd.Series, low: pd.Series) -> pd.Series:
    """
    Corwin–Schultz (2012) bid–ask spread estimator from daily high/low prices.
    Returns the FULL proportional spread, e.g. 0.01 = 1%.
    Input must be two aligned Series for ONE asset.
    """
    high = pd.Series(high).astype(float)
    low = pd.Series(low).astype(float)

    idx = high.dropna().index.intersection(low.dropna().index)
    high = high.loc[idx].sort_index()
    low = low.loc[idx].sort_index()

    hl = np.log(high / low)
    beta = hl.pow(2) + hl.shift(1).pow(2)

    high2 = pd.concat([high, high.shift(1)], axis=1).max(axis=1)
    low2 = pd.concat([low, low.shift(1)], axis=1).min(axis=1)
    gamma = np.log(high2 / low2).pow(2)

    k = 3 - 2 * np.sqrt(2)
    alpha = (np.sqrt(2 * beta) - np.sqrt(beta)) / k - np.sqrt(gamma / k)
    alpha = alpha.clip(lower=0)

    spread = 2 * (np.exp(alpha) - 1) / (1 + np.exp(alpha))
    spread.name = "cs_spread"
    return spread


def corwin_schultz_panel(
    px_high: pd.DataFrame,
    px_low: pd.DataFrame,
    tickers: list | None = None,
    clip_upper: float | None = 0.10,
) -> pd.DataFrame:
    """
    Apply Corwin–Schultz asset by asset on a panel of highs/lows.
    Returns a DataFrame indexed by date, columns=tickers, with FULL spreads.
    """
    px_high = px_high.copy()
    px_low = px_low.copy()

    if tickers is None:
        tickers = [c for c in px_high.columns if c in px_low.columns]

    out = {}
    for t in tickers:
        h = px_high[t].dropna()
        l = px_low[t].dropna()

        idx = h.index.intersection(l.index)
        if len(idx) < 3:
            out[t] = pd.Series(dtype=float)
            continue

        s = corwin_schultz_spread(h.loc[idx], l.loc[idx])

        if clip_upper is not None:
            s = s.clip(upper=clip_upper)

        out[t] = s

    panel = pd.DataFrame(out).sort_index()
    panel = panel.reindex(columns=tickers)
    return panel


def cs_panel_to_one_way_cost(cs_spreads: pd.DataFrame) -> pd.DataFrame:
    """
    Convert FULL spread to ONE-WAY implementation cost.
    If spread = ask-bid over mid, a one-way execution cost is half-spread.
    """
    return 0.5 * cs_spreads.astype(float)

# ============================================================
# 3) Corwin--Schultz asset-level one-way costs
# ============================================================

cs_spreads = corwin_schultz_panel(
    px_high=px_high,
    px_low=px_low,
    tickers=list(R.columns),
    clip_upper=None,
)

asset_costs = (
    cs_panel_to_one_way_cost(cs_spreads)
    .reindex(index=R.index, columns=R.columns)
    .ffill()
    .fillna(0.0)
)

print("One-way Corwin--Schultz cost summary (bps) by asset:")
display((1e4 * asset_costs).describe(percentiles=[0.50, 0.95]).T[["mean", "50%", "95%", "max"]].sort_values("50%"))


## 4) Walk-forward PC1 base strategy and overlay design panel

The overlay operates on the tradable walk-forward PC1 return series. The base PC1 series is **already net of PC rebalance costs** from the Corwin--Schultz one-way cost estimates. Overlay costs are added later as incremental costs from scalar exposure changes.


In [ ]:
# ============================================================
# 4) Walk-forward PC1 and development / OOS overlay panel
# ============================================================

assert "TRAIN_END_DEFAULT" in globals()
assert "TEST_START_DEFAULT" in globals()
assert "START" in globals()
assert "WF_TRAIN_YEARS" in globals()

DEV_END = pd.Timestamp(TRAIN_END_DEFAULT)
OOS_START = pd.Timestamp(TEST_START_DEFAULT)
WF_FIRST_TRADABLE = (pd.Timestamp(START) + pd.DateOffset(years=WF_TRAIN_YEARS)).normalize()

print("Earliest feasible walk-forward start:", WF_FIRST_TRADABLE.date())
print("Development end:", DEV_END.date(), "| OOS start:", OOS_START.date())

F_wf_long, F_wf_net_long, W_hist_long = walk_forward_pca(
    R=R,
    start_test=str(WF_FIRST_TRADABLE.date()),
    train_years=WF_TRAIN_YEARS,
    step_months=WF_STEP_MONTHS,
    k=WF_K_CHOICE,
    gross_norm=GROSS_NORM,
    asset_costs=asset_costs,
    charge_initial_rebalance=False,
)

pc1_long = F_wf_net_long[PC_CORE].dropna().astype(float).copy()
pc1_long.name = "pc1_base"

H_MEDIUM = 21
panel = pd.DataFrame({"pc1_base": pc1_long})

# Realized-volatility features for HAR / OU layers.
panel["rv_5"] = rolling_realized_vol(panel["pc1_base"], window=5, ann=ANN)
panel["rv_21"] = rolling_realized_vol(panel["pc1_base"], window=21, ann=ANN)
panel["rv_63"] = rolling_realized_vol(panel["pc1_base"], window=63, ann=ANN)
panel["dsv_21"] = rolling_downside_semivol(panel["pc1_base"], window=21, ann=ANN)
panel["rv_fwd_H"] = forward_realized_vol_from_returns(panel["pc1_base"], horizon=H_MEDIUM, ann=ANN)

# HAR lagged components.
panel["rv_21_lag1"] = panel["rv_21"].shift(1)
panel["rv_5_lag1"] = panel["rv_5"].shift(1)
panel["rv_21_lag5_mean"] = panel["rv_21"].shift(1).rolling(5, min_periods=5).mean()
panel["rv_21_lag21_mean"] = panel["rv_21"].shift(1).rolling(21, min_periods=15).mean()

panel = panel.dropna().copy()
dev = panel.loc[:DEV_END].copy()
oos = panel.loc[OOS_START:].copy()

print("Long walk-forward PC1:", pc1_long.index.min().date(), "->", pc1_long.index.max().date(), "| n =", len(pc1_long))
print("Panel:", panel.index.min().date(), "->", panel.index.max().date(), "| n =", len(panel))
print("DEV:", dev.index.min().date(), "->", dev.index.max().date(), "| n =", len(dev))
print("OOS:", oos.index.min().date(), "->", oos.index.max().date(), "| n =", len(oos))

print("\nBase PC1 OOS performance:")
display(perf_table({"PC1_base": oos["pc1_base"]}).round(4))

display(panel.head())


## 5) HAR-RV residual-bootstrap medium layer

This layer is retained as a methodological benchmark / medium-speed component. It forecasts the distribution of forward realized volatility with a HAR mean model plus residual bootstrap and maps the exceedance score into a low-turnover exposure rule.


In [ ]:
# ============================================================
# V5 — HAR-RV + residual bootstrap predictive distribution
# Goal: estimate P( RV_fwd_H > tau | information at t )
# using HAR mean dynamics + bootstrapped residual uncertainty
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

# ------------------------------------------------------------
# Preconditions:
# expects dev / oos from the base-panel cell already available
# with columns:
#   rv_fwd_H, rv_5_lag1, rv_21_lag5_mean, rv_21_lag21_mean
# ------------------------------------------------------------
assert "dev" in globals() and "oos" in globals(), "Run the walk-forward PC1 panel cell first so that dev and oos exist."

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BOOT_B = 5000                 # number of bootstrap draws
TAU_MODE = "dev_percentile"   # "dev_percentile" or "absolute"
TAU_PERCENTILE = 0.80         # e.g. 0.80, 0.85, 0.90
TAU_ABS = 0.30                # used only if TAU_MODE == "absolute"

HAR_COLS = ["rv_5_lag1", "rv_21_lag5_mean", "rv_21_lag21_mean"]
Y_COL = "rv_fwd_H"

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def auc_rank(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum())
    n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    auc = (ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0)
    return float(auc)

def brier_score(y_true: pd.Series, p_hat: pd.Series) -> float:
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))

def bucket_prob_calibration(y_true: pd.Series, p_hat: pd.Series, q: int = 5) -> pd.DataFrame:
    tmp = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna().copy()
    if tmp.empty:
        return pd.DataFrame()
    tmp["bucket"] = pd.qcut(tmp["p"], q=q, duplicates="drop")
    out = tmp.groupby("bucket", observed=False).agg(
        n=("y", "size"),
        mean_p=("p", "mean"),
        realized_rate=("y", "mean"),
    )
    out["realized_rate"] *= 100
    out["mean_p"] *= 100
    return out

# ------------------------------------------------------------
# 1) Fit HAR on development sample
# ------------------------------------------------------------
har_dev = dev[[Y_COL] + HAR_COLS].dropna().copy()
X_dev = sm.add_constant(har_dev[HAR_COLS], has_constant="add")
y_dev = har_dev[Y_COL].astype(float)

har_res = sm.OLS(y_dev, X_dev).fit()
har_resid = (y_dev - har_res.fittedvalues).dropna()

print("HAR fit on development sample")
print("n_dev:", len(har_dev))
display(pd.DataFrame({
    "coef": har_res.params,
    "tstat": har_res.tvalues,
    "pvalue": har_res.pvalues,
}))

# ------------------------------------------------------------
# 2) Choose threshold tau
# ------------------------------------------------------------
if TAU_MODE == "dev_percentile":
    tau = float(dev[Y_COL].dropna().quantile(TAU_PERCENTILE))
    tau_label = f"dev p{int(TAU_PERCENTILE*100)}"
else:
    tau = float(TAU_ABS)
    tau_label = f"absolute {TAU_ABS:.4f}"

print(f"\nChosen volatility threshold tau = {tau:.6f} ({tau_label})")

# ------------------------------------------------------------
# 3) OOS predictive distribution via residual bootstrap
#    Frozen-coefficient version first (clean baseline)
# ------------------------------------------------------------
har_oos = oos[[Y_COL] + HAR_COLS].dropna().copy()
X_oos = sm.add_constant(har_oos[HAR_COLS], has_constant="add")
mu_oos = pd.Series(har_res.predict(X_oos), index=har_oos.index, name="mu_hat")

# bootstrap residual draws
resid_pool = har_resid.values.astype(float)
boot_draws = np.random.choice(resid_pool, size=(len(mu_oos), BOOT_B), replace=True)

# predictive draws: mean + resampled residual
pred_draws = mu_oos.values.reshape(-1, 1) + boot_draws

# optional clipping to nonnegative values
pred_draws = np.clip(pred_draws, 1e-8, None)

# predictive summaries
p_exceed = pd.Series((pred_draws > tau).mean(axis=1), index=mu_oos.index, name="p_rv_exceed")
q90 = pd.Series(np.quantile(pred_draws, 0.90, axis=1), index=mu_oos.index, name="rv_q90")
pred_mean = pd.Series(pred_draws.mean(axis=1), index=mu_oos.index, name="rv_pred_mean")
pred_std = pd.Series(pred_draws.std(axis=1, ddof=1), index=mu_oos.index, name="rv_pred_std")

# realized event
event_oos = (har_oos[Y_COL] > tau).astype(int).rename("event_rv_exceed")

# ------------------------------------------------------------
# 4) Diagnostics
# ------------------------------------------------------------
diag = pd.DataFrame({
    "realized_rv_fwd": har_oos[Y_COL],
    "rv_pred_mean": pred_mean,
    "rv_pred_std": pred_std,
    "rv_q90": q90,
    "p_rv_exceed": p_exceed,
    "event_rv_exceed": event_oos,
})

print("\nOOS predictive distribution diagnostics")
summary_diag = pd.DataFrame([{
    "oos_n": len(diag),
    "tau": tau,
    "event_rate_%": 100 * diag["event_rv_exceed"].mean(),
    "mean_pred_p_%": 100 * diag["p_rv_exceed"].mean(),
    "AUC": auc_rank(diag["event_rv_exceed"], diag["p_rv_exceed"]),
    "Brier": brier_score(diag["event_rv_exceed"], diag["p_rv_exceed"]),
}]).T
summary_diag.columns = ["value"]
display(summary_diag)

print("\nProbability calibration by buckets")
display(bucket_prob_calibration(diag["event_rv_exceed"], diag["p_rv_exceed"], q=5))

print("\nTail of OOS predictive table")
display(diag.tail(10))

# ------------------------------------------------------------
# 5) Simple plots
# ------------------------------------------------------------
ax = diag[["realized_rv_fwd", "rv_pred_mean", "rv_q90"]].plot(
    title=f"V5 — OOS HAR predictive distribution | tau={tau:.3f}"
)
ax.set_ylabel("forward realized volatility")

ax2 = diag["p_rv_exceed"].plot(
    title=f"V5 — OOS probability that forward RV exceeds tau={tau:.3f}"
)
ax2.set_ylabel("probability")

# ------------------------------------------------------------
# 6) Save useful objects for next overlay step
# ------------------------------------------------------------
har_boot_oos = diag.copy()
har_boot_oos["tau"] = tau
har_boot_oos["tau_mode"] = tau_label

print("\nSaved object: har_boot_oos")

In [ ]:
# ============================================================
# V5.1 — Medium HAR overlay from bootstrap volatility score
# Weekly rebalance + hysteresis + mild de-risking
# Uses har_boot_oos from V5 and oos from the base-panel cell
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

assert "har_boot_oos" in globals(), "Run the HAR-RV predictive distribution cell first so that har_boot_oos exists."
assert "oos" in globals(), "Run the walk-forward PC1 panel cell first so that oos exists."

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
BASE_COL = "pc1_base"
SCORE_COL = "p_rv_exceed"

# exposure states
EXP_NORMAL = 1.00
EXP_DEF = 0.75
EXP_EXT = 0.50

# hysteresis on expanding percentiles of the score
ENTER_DEF_Q = 0.90
EXIT_DEF_Q  = 0.75
ENTER_EXT_Q = 0.98
EXIT_EXT_Q  = 0.90

MIN_HIST_WEEKS = 26          # before this, stay fully invested
OVERLAY_COST_BPS = 0.0       # can raise later if you want

# ------------------------------------------------------------
# Helper: performance table
# ------------------------------------------------------------
def perf_table(return_dict: dict[str, pd.Series], ann: int = 252) -> pd.DataFrame:
    rows = []
    for name, r in return_dict.items():
        r = pd.Series(r).dropna().astype(float)
        if r.empty:
            continue

        wealth = (1.0 + r).cumprod()
        n = len(r)
        years = n / ann

        cagr = wealth.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
        vol = r.std(ddof=1) * np.sqrt(ann)
        mu = r.mean() * ann
        sharpe = mu / vol if vol > 0 else np.nan

        downside = r[r < 0]
        dvol = downside.std(ddof=1) * np.sqrt(ann) if len(downside) > 1 else np.nan
        sortino = mu / dvol if pd.notna(dvol) and dvol > 0 else np.nan

        peak = wealth.cummax()
        dd = wealth / peak - 1.0
        max_dd = dd.min()
        calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

        ulcer = np.sqrt(np.mean((100 * dd) ** 2)) / 100
        martin = cagr / ulcer if ulcer > 0 else np.nan

        rows.append({
            "n": n,
            "CAGR": cagr,
            "AnnRet": mu,
            "AnnVol": vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "MaxDD": max_dd,
            "Calmar": calmar,
            "Ulcer": ulcer,
            "Martin": martin,
        })

    return pd.DataFrame(rows, index=list(return_dict.keys()))

# ------------------------------------------------------------
# 1) Build daily base series and score series
# ------------------------------------------------------------
base_r = oos[BASE_COL].dropna().astype(float).copy()
score_daily = har_boot_oos[SCORE_COL].dropna().astype(float).copy()

# align on common daily index
common_idx = base_r.index.intersection(score_daily.index)
base_r = base_r.loc[common_idx]
score_daily = score_daily.loc[common_idx]

# weekly rebalance score = last available score each week
score_weekly = score_daily.resample("W-FRI").last().dropna()

# ------------------------------------------------------------
# 2) State machine with expanding percentile thresholds
# ------------------------------------------------------------
state_rows = []
state = "normal"

for i, dt in enumerate(score_weekly.index):
    p_now = float(score_weekly.loc[dt])

    hist = score_weekly.iloc[:i]
    if len(hist) < MIN_HIST_WEEKS:
        q_def_enter = q_def_exit = q_ext_enter = q_ext_exit = np.nan
        state = "normal"
        exposure = EXP_NORMAL
    else:
        q_def_enter = float(hist.quantile(ENTER_DEF_Q))
        q_def_exit  = float(hist.quantile(EXIT_DEF_Q))
        q_ext_enter = float(hist.quantile(ENTER_EXT_Q))
        q_ext_exit  = float(hist.quantile(EXIT_EXT_Q))

        if state == "normal":
            if p_now >= q_ext_enter:
                state = "extreme"
            elif p_now >= q_def_enter:
                state = "defensive"

        elif state == "defensive":
            if p_now >= q_ext_enter:
                state = "extreme"
            elif p_now <= q_def_exit:
                state = "normal"

        elif state == "extreme":
            if p_now <= q_ext_exit:
                # de-escalate with hysteresis
                if p_now <= q_def_exit:
                    state = "normal"
                else:
                    state = "defensive"

        exposure = {
            "normal": EXP_NORMAL,
            "defensive": EXP_DEF,
            "extreme": EXP_EXT,
        }[state]

    state_rows.append({
        "Date": dt,
        "p_score": p_now,
        "q_def_enter": q_def_enter,
        "q_def_exit": q_def_exit,
        "q_ext_enter": q_ext_enter,
        "q_ext_exit": q_ext_exit,
        "state": state,
        "exposure_signal": exposure,
    })

weekly_overlay = pd.DataFrame(state_rows).set_index("Date")

print("Weekly overlay state table")
display(weekly_overlay.tail(15))

# ------------------------------------------------------------
# 3) Push weekly signal to daily frequency
# ------------------------------------------------------------
exposure_signal_daily = weekly_overlay["exposure_signal"].reindex(base_r.index, method="ffill").fillna(EXP_NORMAL)

# apply from next trading day to avoid same-day look-ahead
exposure_applied = exposure_signal_daily.shift(1).fillna(EXP_NORMAL).rename("exposure_applied")

# turnover + optional cost
overlay_turnover = exposure_applied.diff().abs().fillna(0.0).rename("overlay_turnover")
overlay_cost = (OVERLAY_COST_BPS / 10000.0) * overlay_turnover
overlay_r = (exposure_applied * base_r - overlay_cost).rename("overlay_r")

# ------------------------------------------------------------
# 4) Collect diagnostic panel
# ------------------------------------------------------------
overlay_panel = pd.concat(
    [
        base_r.rename("base_r"),
        score_daily.rename("p_score_daily"),
        exposure_signal_daily.rename("exposure_signal_daily"),
        exposure_applied,
        overlay_turnover,
        overlay_cost.rename("overlay_cost"),
        overlay_r,
    ],
    axis=1,
)

print("\nOverlay daily panel tail")
display(overlay_panel.tail(15))

# ------------------------------------------------------------
# 5) Compare performance
# ------------------------------------------------------------
perf = perf_table({
    "PC1_base_OOS": base_r,
    "PC1_HAR_overlay": overlay_r,
})

print("\nPerformance comparison")
display(perf)

# ------------------------------------------------------------
# 6) Event concentration check
#    Do high-score / de-risk states line up with realized RV exceedance events?
# ------------------------------------------------------------
if "event_rv_exceed" in har_boot_oos.columns:
    event_daily = har_boot_oos["event_rv_exceed"].reindex(base_r.index)
    event_check = pd.concat(
        [
            event_daily.rename("event_rv_exceed"),
            score_daily.rename("p_score"),
            exposure_signal_daily.rename("exposure_signal"),
        ],
        axis=1,
    ).dropna()

    state_summary = event_check.groupby("exposure_signal", observed=False).agg(
        n=("event_rv_exceed", "size"),
        mean_p=("p_score", "mean"),
        event_rate=("event_rv_exceed", "mean"),
    )
    state_summary["event_rate"] *= 100

    print("\nEvent concentration by exposure state")
    display(state_summary)

# ------------------------------------------------------------
# 7) Save useful objects
# ------------------------------------------------------------
slow_overlay_oos = overlay_panel.copy()
slow_overlay_weekly = weekly_overlay.copy()

print("\nSaved objects: slow_overlay_oos, slow_overlay_weekly")

## 6) OU-style probabilistic slow layer

This is the retained slow layer. Candidate OU-style overlays are selected by **development-sample AUC of the volatility-exceedance score**, not by OOS Sharpe or P&L.

Caveat: the development window is short and calm; when the exposure state machine does not trade in development, overlay P&L is degenerate. The selection rule therefore ranks the predictive signal rather than development-period overlay returns, and the caveat is reported explicitly.


In [ ]:

# ============================================================
# V6b — OU slow-layer design sweep *(selection fixed)*
# Purpose: horizon / state-variable / threshold sensitivity.
#
# Important correction vs the earlier version:
#   - candidates are NOT selected by OOS Sharpe;
#   - the preferred candidate is selected by DEVELOPMENT AUC of the OU exceedance score;
#   - development overlay performance is reported, but not used when dev_trades=0;
#   - OOS rankings are diagnostics/robustness only.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm
from IPython.display import display
import matplotlib.pyplot as plt

assert "dev" in globals() and "oos" in globals(), "Run the walk-forward PC1 panel cell first."

BASE_COL = "pc1_base"
ANN = 252

# For the sweep only, use a lighter minimum calibration window so that the
# 2017 development sample can actually rank candidates without looking at OOS.
# The main V6 OU layer remains the pre-specified benchmark layer.
MIN_CALIB_OBS = 126
MIN_SELECTION_SCORE_OBS = 30

# Same mild weekly hysteresis as the original slow challenger
SWEEP_ENTER_DEF_Q = 0.85
SWEEP_EXIT_DEF_Q  = 0.70
SWEEP_ENTER_EXT_Q = 0.96
SWEEP_EXIT_EXT_Q  = 0.85
SWEEP_EXP_NORMAL  = 1.00
SWEEP_EXP_DEF     = 0.75
SWEEP_EXP_EXT     = 0.50
MIN_HIST_WEEKS    = 10

HORIZON_GRID = [21, 42, 63]
TAU_GRID = [0.80, 0.85]


def _rolling_rv_sweep(r, window=21, ann=252):
    return (
        pd.Series(r).astype(float)
        .rolling(window, min_periods=max(5, window // 2))
        .std(ddof=1) * np.sqrt(ann)
    ).rename(f"rv_{window}")


def _rolling_dsv_sweep(r, window=21, ann=252, mar=0.0):
    r = pd.Series(r).astype(float)
    neg = np.minimum(r - mar, 0.0)
    return np.sqrt(
        ann * pd.Series(neg**2, index=r.index).rolling(window, min_periods=max(5, window // 2)).mean()
    ).rename(f"dsv_{window}")


def fit_ou_positive_series(x):
    x = pd.Series(x).dropna().astype(float)
    x = x[x > 0]
    if len(x) < 30:
        return None
    logx = np.log(x)
    y = logx.iloc[1:].values
    X = sm.add_constant(logx.iloc[:-1].values, has_constant="add")
    try:
        res = sm.OLS(y, X).fit()
    except Exception:
        return None
    alpha, beta = float(res.params[0]), float(res.params[1])
    xi = float(np.std(res.resid, ddof=2))
    if not np.isfinite(beta) or beta <= 0 or beta >= 1:
        return None
    kappa = -np.log(beta)
    theta = alpha / (1.0 - beta)
    return {"alpha": alpha, "beta": beta, "xi": xi, "kappa": kappa, "theta": theta}


def ou_predictive(log_x_t, params, H):
    kappa, theta, xi = params["kappa"], params["theta"], params["xi"]
    exp_kH = np.exp(-kappa * H)
    mu_H   = theta + (log_x_t - theta) * exp_kH
    var_H  = (xi ** 2) / (2.0 * kappa) * (1.0 - np.exp(-2.0 * kappa * H))
    std_H  = np.sqrt(max(var_H, 1e-12))
    return mu_H, std_H


def auc_rank(y_true, p_hat):
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return np.nan
    y = df["y"].astype(int).values
    p = df["p"].astype(float).values
    ranks = pd.Series(p).rank(method="average").values
    n1 = int(y.sum()); n0 = int(len(y) - n1)
    if n1 == 0 or n0 == 0:
        return np.nan
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def brier_score(y_true, p_hat):
    df = pd.concat([pd.Series(y_true, name="y"), pd.Series(p_hat, name="p")], axis=1).dropna()
    if df.empty:
        return np.nan
    return float(np.mean((df["p"].astype(float) - df["y"].astype(float)) ** 2))


def _perf(r, ann=252):
    r = pd.Series(r).dropna().astype(float)
    if len(r) < 5:
        return {"n": len(r), "CAGR": np.nan, "AnnVol": np.nan, "Sharpe": np.nan,
                "Sortino": np.nan, "MaxDD": np.nan, "Calmar": np.nan, "Martin": np.nan}
    w = (1 + r).cumprod()
    n = len(r); yrs = n / ann
    cagr = w.iloc[-1]**(1/yrs)-1 if yrs > 0 else np.nan
    vol = r.std(ddof=1) * np.sqrt(ann)
    mu = r.mean() * ann
    shr = mu / vol if vol > 0 else np.nan
    dn = r[r < 0]
    dvol = dn.std(ddof=1) * np.sqrt(ann) if len(dn) > 1 else np.nan
    srt = mu / dvol if pd.notna(dvol) and dvol > 0 else np.nan
    pk = w.cummax(); dd = w / pk - 1; mdd = dd.min()
    cal = cagr / abs(mdd) if mdd < 0 else np.nan
    ulc = np.sqrt(np.mean((100 * dd) ** 2)) / 100
    mar = cagr / ulc if ulc > 0 else np.nan
    return {"n": n, "CAGR": cagr, "AnnVol": vol, "Sharpe": shr, "Sortino": srt,
            "MaxDD": mdd, "Calmar": cal, "Martin": mar}


def make_ou_scores(eval_index, state_full, H, tau, name):
    scores = pd.Series(np.nan, index=eval_index, name=name)
    for i, dt in enumerate(eval_index):
        hist = state_full.loc[:dt].dropna()
        if len(hist) < MIN_CALIB_OBS:
            continue
        params = fit_ou_positive_series(hist)
        if params is None:
            continue
        x_now = float(hist.iloc[-1])
        mu_H, std_H = ou_predictive(np.log(x_now), params, H=H)
        z = (np.log(tau) - mu_H) / std_H
        scores.loc[dt] = float(1.0 - norm.cdf(z))
    return scores


def build_weekly_overlay_from_scores(scores, base_r):
    base_r = pd.Series(base_r).dropna().astype(float)
    score_weekly = pd.Series(scores).dropna().resample("W-FRI").last().dropna()
    state = "normal"
    rows = []
    for i, dt in enumerate(score_weekly.index):
        p_now = float(score_weekly.loc[dt])
        hist = score_weekly.iloc[:i]
        if len(hist) < MIN_HIST_WEEKS or np.isnan(p_now):
            state = "normal"
            exposure = SWEEP_EXP_NORMAL
        else:
            q_de = float(hist.quantile(SWEEP_ENTER_DEF_Q))
            q_dx = float(hist.quantile(SWEEP_EXIT_DEF_Q))
            q_ee = float(hist.quantile(SWEEP_ENTER_EXT_Q))
            q_ex = float(hist.quantile(SWEEP_EXIT_EXT_Q))
            if state == "normal":
                if p_now >= q_ee:
                    state = "extreme"
                elif p_now >= q_de:
                    state = "defensive"
            elif state == "defensive":
                if p_now >= q_ee:
                    state = "extreme"
                elif p_now < q_dx:
                    state = "normal"
            elif state == "extreme":
                if p_now < q_ex:
                    state = "defensive"
            exposure = {"normal": SWEEP_EXP_NORMAL, "defensive": SWEEP_EXP_DEF, "extreme": SWEEP_EXP_EXT}[state]
        rows.append({"date": dt, "state": state, "score": p_now, "exposure": exposure})

    if not rows:
        exp_signal = pd.Series(SWEEP_EXP_NORMAL, index=base_r.index, name="exposure_signal")
    else:
        weekly = pd.DataFrame(rows).set_index("date")
        exp_signal = weekly["exposure"].reindex(base_r.index, method="ffill").fillna(SWEEP_EXP_NORMAL).astype(float)
        exp_signal.name = "exposure_signal"

    # Apply signal with a one-trading-day lag.
    exp_applied = exp_signal.shift(1).fillna(SWEEP_EXP_NORMAL).rename("exposure_applied")
    overlay_r = base_r * exp_applied
    trades = int((exp_applied.diff().abs() > 0).sum())
    return exp_signal, exp_applied, overlay_r, trades


# Prefer the full walk-forward PC1 series if it is still in memory; this gives
# the development sweep the earliest possible pre-2018 history without using OOS
# to select candidates.
pc1_source = globals().get("pc1_long", None)
if pc1_source is None:
    pc1_source = pd.concat([dev[BASE_COL], oos[BASE_COL]]).sort_index().drop_duplicates()
else:
    pc1_source = pd.Series(pc1_source).sort_index().drop_duplicates()

base_r_dev = dev[BASE_COL].dropna().astype(float)
base_r_oos = oos[BASE_COL].dropna().astype(float)

state_builders = {
    "RV21":  lambda r: _rolling_rv_sweep(r, window=21, ann=ANN),
    "RV63":  lambda r: _rolling_rv_sweep(r, window=63, ann=ANN),
    "DSV21": lambda r: _rolling_dsv_sweep(r, window=21, ann=ANN),
    "DSV63": lambda r: _rolling_dsv_sweep(r, window=63, ann=ANN),
}

rows = []
best_key_dev = None
best_dev_auc = -np.inf
best_bundle_dev = None

for state_name, builder in state_builders.items():
    state_full = builder(pc1_source).dropna()
    state_dev = state_full.reindex(base_r_dev.index).dropna()
    if state_dev.empty:
        continue

    for H in HORIZON_GRID:
        for tau_q in TAU_GRID:
            tau = float(state_dev.quantile(tau_q))
            score_name = f"p_{state_name}_H{H}_q{int(tau_q*100)}"

            scores_dev = make_ou_scores(base_r_dev.index, state_full, H, tau, score_name)
            exp_sig_dev, exp_app_dev, overlay_dev, trades_dev = build_weekly_overlay_from_scores(scores_dev, base_r_dev)
            perf_dev = _perf(overlay_dev)

            scores_oos = make_ou_scores(base_r_oos.index, state_full, H, tau, score_name)
            exp_sig_oos, exp_app_oos, overlay_oos, trades_oos = build_weekly_overlay_from_scores(scores_oos, base_r_oos)
            perf_oos = _perf(overlay_oos)

            # Selection diagnostics: evaluate the SIGNAL on development, not the
            # overlay P&L. This avoids a degenerate selection when the dev
            # exposure state machine never trades.
            realized_future_dev = state_full.shift(-H).reindex(base_r_dev.index)
            event_dev = (realized_future_dev > tau).astype(float)
            auc_dev = auc_rank(event_dev, scores_dev)
            brier_dev = brier_score(event_dev, scores_dev)

            # OOS diagnostics only. Do not use for model selection.
            realized_future_oos = state_full.shift(-H).reindex(base_r_oos.index)
            event_oos = (realized_future_oos > tau).astype(float)
            auc_oos = auc_rank(event_oos, scores_oos)
            brier_oos = brier_score(event_oos, scores_oos)

            n_score_dev = int(scores_dev.notna().sum())
            row = {
                "state_var": state_name,
                "H": H,
                "tau_q": tau_q,
                "tau": tau,
                "dev_score_obs": n_score_dev,
                "dev_trades": trades_dev,
                "Dev_CAGR": perf_dev["CAGR"],
                "Dev_Sharpe": perf_dev["Sharpe"],
                "Dev_MaxDD": perf_dev["MaxDD"],
                "Dev_AUC": auc_dev,
                "Dev_Brier": brier_dev,
                "dev_event_rate": float(event_dev.mean()),
                "oos_score_mean": float(scores_oos.mean()),
                "oos_event_rate": float(event_oos.mean()),
                "OOS_AUC": auc_oos,
                "OOS_Brier": brier_oos,
                "oos_trades": trades_oos,
                "OOS_CAGR": perf_oos["CAGR"],
                "OOS_Sharpe": perf_oos["Sharpe"],
                "OOS_Sortino": perf_oos["Sortino"],
                "OOS_MaxDD": perf_oos["MaxDD"],
                "OOS_Calmar": perf_oos["Calmar"],
                "OOS_Martin": perf_oos["Martin"],
            }
            rows.append(row)

            valid_for_selection = (
                n_score_dev >= MIN_SELECTION_SCORE_OBS
                and np.isfinite(auc_dev)
            )
            if valid_for_selection and auc_dev > best_dev_auc:
                best_dev_auc = auc_dev
                best_key_dev = (state_name, H, tau_q)
                best_bundle_dev = {
                    "scores_dev": scores_dev,
                    "scores_oos": scores_oos,
                    "event_dev": event_dev,
                    "event_oos": event_oos,
                    "exp_signal_dev": exp_sig_dev,
                    "exp_applied_dev": exp_app_dev,
                    "exp_signal_oos": exp_sig_oos,
                    "exp_applied_oos": exp_app_oos,
                    "overlay_dev": overlay_dev,
                    "overlay_oos": overlay_oos,
                    "state_full": state_full,
                    "tau": tau,
                    "perf_dev": perf_dev,
                    "perf_oos": perf_oos,
                }

ou_sweep = pd.DataFrame(rows)
if not ou_sweep.empty:
    if (ou_sweep["dev_trades"] == 0).all():
        print("WARNING: development overlay P&L is degenerate: dev_trades=0 for every candidate. Selecting by Dev_AUC of the OU exceedance score instead of Dev Sharpe.")
    ou_sweep = ou_sweep.sort_values(
        ["Dev_AUC", "Dev_Brier", "Dev_Sharpe", "OOS_Sharpe"],
        ascending=[False, True, False, False]
    ).reset_index(drop=True)

print("OU slow-layer design sweep — ranked by DEVELOPMENT AUC of the exceedance score")
display(ou_sweep.head(12).round(4))

print("\nOOS diagnostic ranking only — do not use this for model selection")
display(
    ou_sweep.sort_values(["OOS_Sharpe", "OOS_Martin", "OOS_Calmar"], ascending=[False, False, False])
    .head(12).round(4)
)

if best_bundle_dev is not None:
    print(
        f"Selected OU-style challenger by DEV AUC: "
        f"state={best_key_dev[0]}, H={best_key_dev[1]}, tau_q={best_key_dev[2]:.2f}"
    )
    cmp_selected = pd.DataFrame({
        "PC1_base_DEV": _perf(base_r_dev),
        "Selected_OU_DEV": best_bundle_dev["perf_dev"],
        "PC1_base_OOS": _perf(base_r_oos),
        "Selected_OU_OOS": best_bundle_dev["perf_oos"],
    }).T
    display(cmp_selected.round(4))

    ou_sweep_selected = pd.DataFrame({
        "base_r": base_r_oos,
        "score": best_bundle_dev["scores_oos"].reindex(base_r_oos.index),
        "event_proxy": best_bundle_dev["event_oos"].reindex(base_r_oos.index),
        "exposure_signal": best_bundle_dev["exp_signal_oos"].reindex(base_r_oos.index),
        "exposure_applied": best_bundle_dev["exp_applied_oos"].reindex(base_r_oos.index),
        "overlay_r": best_bundle_dev["overlay_oos"].reindex(base_r_oos.index),
    })
    ou_sweep_selected.attrs["selection_rule"] = "max DEV AUC of OU exceedance score, not OOS Sharpe"
    ou_sweep_selected.attrs["selection_caveat"] = globals().get("ou_sweep_selection_caveat", "Dev-AUC selection; disclose development-window limitations.")
    ou_sweep_selected.attrs["selected_state_var"] = best_key_dev[0]
    ou_sweep_selected.attrs["selected_H"] = best_key_dev[1]
    ou_sweep_selected.attrs["selected_tau_q"] = best_key_dev[2]
    ou_sweep_selected.attrs["selected_tau"] = best_bundle_dev["tau"]

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    (1 + base_r_oos).cumprod().plot(ax=axes[0], label="PC1_base")
    (1 + best_bundle_dev["overlay_oos"]).cumprod().plot(ax=axes[0], label="Selected_OU_OOS")
    axes[0].set_title(f"DEV-selected OU challenger — {best_key_dev[0]}, H={best_key_dev[1]}, tau_q={best_key_dev[2]:.2f}")
    axes[0].legend()

    best_bundle_dev["scores_oos"].dropna().plot(ax=axes[1], label="OU-style score")
    axes[1].set_ylabel("probability")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    print("Saved: ou_sweep, ou_sweep_selected")
else:
    ou_sweep_selected = None
    print(
        "No OU sweep candidate had enough development-sample score observations/events "
        f"(min={MIN_SELECTION_SCORE_OBS}) to select by DEV AUC. OOS sweep is diagnostic only; "
        "do not report a best-OOS OU model."
    )
    print("Saved: ou_sweep; ou_sweep_selected=None")



## 7) Fast tactical risk layer

The current fast layer is a conservative IV-based tactical override. It is included as an implemented challenger / architecture component. A Kalman innovation / Mahalanobis surprise layer is the natural next replacement candidate.


In [ ]:
# ============================================================
# V7 — Fast overlay: IV-based logit tactical override
# Horizon: 5 trading days (1 week)
# Signal: log_iv_{t-1}, dlog_iv^+_t, dlog_iv^-_t
#
# Design:
#   - Fit binomial logit: event = DD_{t,t+5} < DD_THR_FAST
#   - Estimate P(event | F_t) walk-forward on OOS
#   - Override fires ONLY when score > extreme threshold
#   - Exposure: 1.0 (normal) | 0.50 (override active)
#
# Conservative by design: fires rarely, avoids noise-driven turnover.
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
import matplotlib.pyplot as plt

assert "oos" in globals(), "Run the walk-forward PC1 panel cell first."
assert "dev" in globals(), "Run the walk-forward PC1 panel cell first."
assert "TRAIN_END_DEFAULT" in globals(), "TRAIN_END_DEFAULT must be defined by the parameters cell."
assert "TEST_START_DEFAULT" in globals(), "TEST_START_DEFAULT must be defined by the parameters cell."

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
H_FAST          = 5            # forecast horizon (1 week)
DD_THR_FAST     = -0.03        # event: worst 5-day return < -3%
ANN             = 252
BASE_COL        = "pc1_base"
MIN_CALIB_FAST  = 126          # ~6 months before first OOS calibration
HAC_LAGS_FAST   = H_FAST

# Score thresholds (expanding percentiles of OOS score)
FAST_ENTER_Q    = 0.96         # very high threshold — rare override
FAST_EXIT_Q     = 0.88

FAST_EXP_OVERRIDE = 0.50
FAST_EXP_NORMAL   = 1.00

FEAT_COLS = ["log_iv_lag1", "dlog_iv_plus", "dlog_iv_minus"]

# ------------------------------------------------------------
# Helper: build IV predictor panel with asymmetric changes
# ------------------------------------------------------------
def build_iv_fast_panel(iv_level, target_index):
    iv      = pd.Series(iv_level).astype(float).reindex(target_index).ffill()
    log_iv  = np.log(iv)
    dlog_iv = log_iv.diff()
    return pd.DataFrame({
        "log_iv_lag1":    log_iv.shift(1),
        "dlog_iv_plus":   np.maximum(dlog_iv, 0.0),
        "dlog_iv_minus":  np.minimum(dlog_iv, 0.0),
        "d2log_iv":       dlog_iv.diff(),          # acceleration (optional)
    }, index=target_index)


# ------------------------------------------------------------
# Helper: forward worst return over H days
# ------------------------------------------------------------
def fwd_worst(r, H=5):
    r  = pd.Series(r).dropna().astype(float)
    w  = (1.0 + r).cumprod().values
    out = np.full(len(w), np.nan)
    for i in range(len(w)):
        j = min(len(w), i + H + 1)
        if i + 1 < j:
            out[i] = float(np.min(w[i+1:j] / w[i] - 1.0))
    return pd.Series(out, index=r.index, name=f"worst_fwd_{H}d")


# ------------------------------------------------------------
# Helper: HAC logit
# ------------------------------------------------------------
def fit_logit_hac(y, X, hac_lags=5):
    df = pd.concat([pd.Series(y, name="y"), pd.DataFrame(X)], axis=1).dropna()
    if df.empty or df["y"].nunique() < 2:
        return None
    yv = df["y"].astype(float)
    Xv = sm.add_constant(df.drop(columns="y").astype(float), has_constant="add")
    try:
        res = sm.GLM(yv, Xv, family=sm.families.Binomial()).fit(
            cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    except Exception:
        try:
            res = sm.GLM(yv, Xv, family=sm.families.Binomial()).fit()
        except Exception:
            return None
    return res


# ------------------------------------------------------------
# 1) Build full IV panel and targets
# ------------------------------------------------------------
pc1_full_fast = (
    pd.concat([dev[BASE_COL], oos[BASE_COL]])
    .sort_index().drop_duplicates()
)
iv_panel_full = build_iv_fast_panel(iv, pc1_full_fast.index)
worst_full    = fwd_worst(pc1_full_fast, H=H_FAST)
event_full    = (worst_full < DD_THR_FAST).astype(int).rename("event_fast")

df_full_fast = pd.concat([
    pc1_full_fast.rename(BASE_COL),
    worst_full, event_full,
    iv_panel_full[FEAT_COLS],
], axis=1).dropna()

print(f"Fast overlay | H={H_FAST}d | DD threshold: {DD_THR_FAST:.1%}")
print(f"Full sample event rate: {100*df_full_fast['event_fast'].mean():.2f}%")

# Dev in-sample diagnostics
TRAIN_END_TS = pd.Timestamp(TRAIN_END_DEFAULT)
dev_fast = df_full_fast.loc[:TRAIN_END_TS].copy()
# At the end of the development sample, the final H_FAST forward labels would
# not yet be observable. Drop them for model diagnostics/training consistency.
dev_fast_model = dev_fast.iloc[:-H_FAST].copy() if len(dev_fast) > H_FAST else dev_fast.copy()
print(f"Dev sample: {len(dev_fast)} obs | event rate: {100*dev_fast['event_fast'].mean():.2f}%")
print(f"Dev model sample after dropping unobservable final {H_FAST} forward labels: {len(dev_fast_model)} obs")

res_dev_fast = fit_logit_hac(dev_fast_model["event_fast"], dev_fast_model[FEAT_COLS], hac_lags=HAC_LAGS_FAST)
if res_dev_fast is not None:
    tab_dev = pd.DataFrame({
        "coef":       res_dev_fast.params,
        "zstat":      res_dev_fast.tvalues,
        "pvalue":     res_dev_fast.pvalues,
        "odds_ratio": np.exp(res_dev_fast.params),
    })
    print("\nDev-sample logit (fast overlay predictors):")
    display(tab_dev.round(4))
    pseudo_r2 = 1.0 - res_dev_fast.llf / res_dev_fast.llnull
    print(f"McFadden pseudo-R2: {pseudo_r2:.4f}")

# ------------------------------------------------------------
# 2) Walk-forward OOS score generation
# ------------------------------------------------------------
OOS_START_TS = pd.Timestamp(TEST_START_DEFAULT)
oos_fast_idx = df_full_fast.loc[OOS_START_TS:].index
scores_fast  = pd.Series(np.nan, index=oos_fast_idx, name="p_fast")

for i, dt in enumerate(oos_fast_idx):
    # Real-time correction: event_fast at date s uses returns from s+1...s+H_FAST.
    # At date dt those labels are not observed for the most recent H_FAST rows.
    hist = df_full_fast.loc[:dt].iloc[:-(H_FAST + 1)]
    if len(hist) < MIN_CALIB_FAST or hist["event_fast"].nunique() < 2:
        continue
    res_wf = fit_logit_hac(hist["event_fast"], hist[FEAT_COLS], hac_lags=HAC_LAGS_FAST)
    if res_wf is None:
        continue
    row_t  = np.concatenate([[1.0], df_full_fast.loc[dt, FEAT_COLS].values.astype(float)])
    logit_v = float(np.dot(res_wf.params.values, row_t))
    scores_fast.loc[dt] = float(1.0 / (1.0 + np.exp(-logit_v)))

print(f"\nFast scores: {scores_fast.notna().sum()} / {len(oos_fast_idx)} OOS observations")

scores_fast.dropna().plot(
    title=f"Fast overlay score — P(DD_fwd_{H_FAST}d < {DD_THR_FAST:.0%} | IV signals)",
    ylabel="probability", figsize=(10, 3)
)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 3) Daily state machine (binary, high threshold)
# ------------------------------------------------------------
base_r_fast  = oos[BASE_COL].reindex(oos_fast_idx).dropna().astype(float)
state_fast   = "normal"
state_rows_f = []

for dt in oos_fast_idx:
    p_now = float(scores_fast.get(dt, np.nan))
    hist  = scores_fast.loc[:dt].iloc[:-1].dropna()

    if len(hist) < 10 or np.isnan(p_now):
        state_fast = "normal"
        exposure_f = FAST_EXP_NORMAL
    else:
        q_enter = float(hist.quantile(FAST_ENTER_Q))
        q_exit  = float(hist.quantile(FAST_EXIT_Q))
        if state_fast == "normal" and p_now >= q_enter:
            state_fast = "override"
        elif state_fast == "override" and p_now < q_exit:
            state_fast = "normal"
        exposure_f = FAST_EXP_OVERRIDE if state_fast == "override" else FAST_EXP_NORMAL

    state_rows_f.append({"date": dt, "state": state_fast,
                         "exposure": exposure_f, "score": p_now})

state_df_fast = pd.DataFrame(state_rows_f).set_index("date")

# Signal exposure is known only after today's IV/return information is observed.
# Apply it from the next trading day to avoid same-close look-ahead.
exp_signal_daily_fast = (
    state_df_fast["exposure"]
    .reindex(base_r_fast.index).fillna(FAST_EXP_NORMAL).astype(float)
    .rename("exposure_signal_daily")
)
exp_daily_fast = exp_signal_daily_fast.shift(1).fillna(FAST_EXP_NORMAL).rename("exposure_applied")

overlay_r_fast = base_r_fast * exp_daily_fast

trades_fast     = int((exp_daily_fast.diff().abs() > 0).sum())
n_override_days = int((exp_daily_fast < FAST_EXP_NORMAL).sum())
print(f"Fast overlay: {trades_fast} trades | {n_override_days} override days "
      f"/ {len(base_r_fast)} total OOS days")

def _perf(r, ann=252):
    r = pd.Series(r).dropna().astype(float)
    w=(1+r).cumprod(); n=len(r); yrs=n/ann
    cagr=w.iloc[-1]**(1/yrs)-1 if yrs>0 else np.nan
    vol=r.std(ddof=1)*np.sqrt(ann); mu=r.mean()*ann
    shr=mu/vol if vol>0 else np.nan
    dn=r[r<0]; dvol=dn.std(ddof=1)*np.sqrt(ann) if len(dn)>1 else np.nan
    srt=mu/dvol if pd.notna(dvol) and dvol>0 else np.nan
    pk=w.cummax(); dd=w/pk-1; mdd=dd.min()
    cal=cagr/abs(mdd) if mdd<0 else np.nan
    ulc=np.sqrt(np.mean((100*dd)**2))/100
    mar=cagr/ulc if ulc>0 else np.nan
    return {"n":n,"CAGR":cagr,"AnnVol":vol,"Sharpe":shr,
            "Sortino":srt,"MaxDD":mdd,"Calmar":cal,"Martin":mar}

cmp_fast = pd.DataFrame({
    "PC1_base":       _perf(base_r_fast),
    "Fast_IV_gross":  _perf(overlay_r_fast),
}).T
print("\nPerformance (GROSS):")
display(cmp_fast.round(4))

ec_f = pd.DataFrame({"exposure": exp_daily_fast, "base_r": base_r_fast}).dropna()
ec_f["neg_day"] = (ec_f["base_r"] < -0.01).astype(int)
print("\nEvent concentration by fast exposure state:")
display(ec_f.groupby("exposure").agg(
    n=("base_r","size"), mean_ret=("base_r","mean"), neg_rate=("neg_day","mean")
).round(4))

# Save
fast_overlay = {
    "base_r":         base_r_fast,
    "overlay_r":      overlay_r_fast,
    "exposure_signal_daily": exp_signal_daily_fast,
    "exposure_daily":        exp_daily_fast,   # applied, one-trading-day lag
    "score_daily":           scores_fast,
    "state_df":       state_df_fast,
    "dd_threshold":   DD_THR_FAST,
    "horizon":        H_FAST,
}
print("\nSaved: fast_overlay")



## 8) Final validation: net costs, benchmarks, and bootstrap inference

Final comparison includes:
- PC1 base;
- selected OU slow overlay;
- HAR medium overlay;
- fast IV tactical overlay;
- combined overlay using the minimum exposure across layers;
- QQQ and equal-weight universe benchmarks.

All overlay net returns use incremental Corwin--Schultz costs from changes in scalar PC1 exposure.


In [ ]:
# ============================================================
# 8) Final validation table
# Combination rule: exposure_combined = min(selected OU, HAR, fast)
# ============================================================

assert "ou_sweep_selected" in globals() and isinstance(ou_sweep_selected, pd.DataFrame) and not ou_sweep_selected.empty, "Run the OU sweep and select a retained OU layer first."
assert "slow_overlay_oos" in globals(), "Run the HAR overlay cell first."
assert "fast_overlay" in globals(), "Run the fast overlay cell first."
assert "W_hist_long" in globals(), "Run the walk-forward PC1 panel cell first."
assert "asset_costs" in globals(), "Run the Corwin--Schultz cost cell first."
assert "px_stocks" in globals(), "px_stocks not found."

BOOT_B_FINAL = 2000
BLOCK_LEN = int(globals().get("BOOTSTRAP_BLOCK_LEN", 90))
BLOCK_METHOD = globals().get("BOOTSTRAP_BLOCK_METHOD", "fallback fixed value")
COST_SOURCE = "CS"

def _sharpe(r, ann=252):
    r = np.asarray(r, dtype=float)
    r = r[~np.isnan(r)]
    return float(r.mean() / r.std(ddof=1) * np.sqrt(ann)) if len(r) > 10 and r.std(ddof=1) > 0 else np.nan


def _maxdd(r):
    r = np.asarray(r, dtype=float)
    r = r[~np.isnan(r)]
    if len(r) < 2:
        return np.nan
    w = np.cumprod(1.0 + r)
    return float(np.min(w / np.maximum.accumulate(w) - 1.0))


def bootstrap_compare(base, target, label, n_boot=2000, block_len=90, seed=42):
    both = pd.DataFrame({"base": base, "target": target}).dropna()
    arr = both.values
    d_sh, d_mdd = [], []
    try:
        from arch.bootstrap import StationaryBootstrap
        bs = StationaryBootstrap(block_len, arr)
        for b_data, _ in bs.bootstrap(n_boot):
            b = b_data[0] if isinstance(b_data, tuple) else b_data
            if hasattr(b, "values"):
                b = b.values
            d_sh.append(_sharpe(b[:, 1]) - _sharpe(b[:, 0]))
            d_mdd.append(_maxdd(b[:, 1]) - _maxdd(b[:, 0]))
    except ImportError:
        rng = np.random.default_rng(seed)
        n = len(arr)
        print("arch not available — using local Politis--Romano stationary-bootstrap indices")
        for _ in range(n_boot):
            idx = stationary_bootstrap_indices(n, float(block_len), rng)
            b = arr[idx]
            d_sh.append(_sharpe(b[:, 1]) - _sharpe(b[:, 0]))
            d_mdd.append(_maxdd(b[:, 1]) - _maxdd(b[:, 0]))

    d_sh = np.asarray(d_sh, dtype=float)
    d_mdd = np.asarray(d_mdd, dtype=float)
    summary = pd.DataFrame({
        "comparison": [label, label],
        "metric": ["Delta Sharpe", "Delta MaxDD"],
        "mean": [float(np.nanmean(d_sh)), float(np.nanmean(d_mdd))],
        "p05": [float(np.nanpercentile(d_sh, 5)), float(np.nanpercentile(d_mdd, 5))],
        "p50": [float(np.nanmedian(d_sh)), float(np.nanmedian(d_mdd))],
        "p95": [float(np.nanpercentile(d_sh, 95)), float(np.nanpercentile(d_mdd, 95))],
        "P(target_wins)": [float(np.mean(d_sh > 0)), float(np.mean(d_mdd > 0))],
    }).set_index(["comparison", "metric"])
    return summary, d_sh, d_mdd


# ------------------------------------------------------------
# 1) Align exposures
# ------------------------------------------------------------
base_r = ou_sweep_selected["base_r"].dropna().astype(float)
common_idx = base_r.index

exp_slow = ou_sweep_selected["exposure_applied"].reindex(common_idx).fillna(1.0).astype(float)
exp_medium = slow_overlay_oos["exposure_applied"].reindex(common_idx).fillna(1.0).astype(float)
exp_fast = fast_overlay["exposure_daily"].reindex(common_idx).fillna(1.0).astype(float)

exp_combined = pd.concat([exp_slow, exp_medium, exp_fast], axis=1).min(axis=1)
exp_combined.name = "exposure_combined"

print(
    "Retained slow OU candidate:",
    f"state={ou_sweep_selected.attrs.get('selected_state_var')},",
    f"H={ou_sweep_selected.attrs.get('selected_H')},",
    f"tau_q={ou_sweep_selected.attrs.get('selected_tau_q')}"
)
print("Selection caveat:", ou_sweep_selected.attrs.get("selection_caveat", "development-window selection caveat not set"))

# Gross overlay returns.
r_base = base_r.copy()
r_slow_gross = base_r * exp_slow
r_medium_gross = base_r * exp_medium
r_fast_gross = base_r * exp_fast
r_combined_gross = base_r * exp_combined

# ------------------------------------------------------------
# 2) Incremental Corwin--Schultz overlay costs
# ------------------------------------------------------------
tickers = list(px_stocks.columns)
cost_panel = asset_costs.reindex(index=common_idx, columns=tickers).ffill().fillna(0.0).astype(float)
W_pc_daily = factor_weights_daily_from_history(W_hist_long, PC_CORE, common_idx, tickers)
gross_abs = W_pc_daily.abs().sum(axis=1).replace(0.0, np.nan)
portfolio_oneway_cost = ((W_pc_daily.abs() * cost_panel).sum(axis=1) / gross_abs).fillna(0.0)
portfolio_oneway_cost.name = "portfolio_oneway_cost_CS"

def overlay_net_from_exposure(name, r_gross, exp):
    delta = pd.Series(exp).reindex(common_idx).diff().abs().fillna(0.0)
    tc = (delta * portfolio_oneway_cost).rename(f"tc_{name}_CS")
    r_net = (pd.Series(r_gross).reindex(common_idx) - tc).rename(f"{name}_net_CS")
    diag = {
        "strategy": name,
        "avg_oneway_cost_bps": 1e4 * portfolio_oneway_cost.mean(),
        "p95_oneway_cost_bps": 1e4 * portfolio_oneway_cost.quantile(0.95),
        "avg_overlay_turnover": float(delta.mean()),
        "total_overlay_cost_%": float(100 * tc.sum()),
        "days_with_overlay_trade": int((delta > 0).sum()),
    }
    return r_net, tc, delta, diag

r_slow_net, tc_slow, delta_slow, diag_slow = overlay_net_from_exposure("Slow_OU_selected", r_slow_gross, exp_slow)
r_medium_net, tc_medium, delta_medium, diag_medium = overlay_net_from_exposure("Medium_HAR", r_medium_gross, exp_medium)
r_fast_net, tc_fast, delta_fast, diag_fast = overlay_net_from_exposure("Fast_IV", r_fast_gross, exp_fast)
r_combined_net, tc_combined, delta_combined, diag_combined = overlay_net_from_exposure("Combined_selectedOU", r_combined_gross, exp_combined)

cost_diag_layers = pd.DataFrame([diag_slow, diag_medium, diag_fast, diag_combined]).set_index("strategy")
print("\nOverlay incremental cost diagnostics by layer")
display(cost_diag_layers.round(4))

# ------------------------------------------------------------
# 3) Benchmarks and aligned performance table
# ------------------------------------------------------------
qqq_r = pd.Series(B_ret).dropna().astype(float).rename("QQQ")
ew_r = R[TICKERS_US_TECH].mean(axis=1).dropna().astype(float).rename("EW_universe_gross")
benchmark_idx = common_idx.intersection(qqq_r.index).intersection(ew_r.index)

series = {
    "PC1_base": r_base,
    "Slow_OU_selected_gross": r_slow_gross,
    "Slow_OU_selected_net_CS": r_slow_net,
    "Medium_HAR_gross": r_medium_gross,
    "Medium_HAR_net_CS": r_medium_net,
    "Fast_IV_gross": r_fast_gross,
    "Fast_IV_net_CS": r_fast_net,
    "Combined_selectedOU_gross": r_combined_gross,
    "Combined_selectedOU_net_CS": r_combined_net,
    "QQQ": qqq_r,
    "EW_universe_gross": ew_r,
}
series = {k: pd.Series(v).reindex(benchmark_idx).dropna() for k, v in series.items()}
results = perf_table(series, ann=ANN)

print("\n=== Full performance comparison, aligned dates ===")
print(f"Aligned sample: {benchmark_idx.min().date()} -> {benchmark_idx.max().date()} | n={len(benchmark_idx)}")
display(results.round(4))

# ------------------------------------------------------------
# 4) Binding layer analysis
# ------------------------------------------------------------
slow_binding = int(((exp_combined == exp_slow) & (exp_slow < 1.0)).sum())
medium_binding = int(((exp_combined == exp_medium) & (exp_medium < 1.0)).sum())
fast_binding = int(((exp_combined == exp_fast) & (exp_fast < 1.0)).sum())
all_normal = int((exp_combined == 1.0).sum())
N = len(common_idx)

print(f"\nBinding layer analysis ({N} OOS days):")
print(f"  Normal (exp=1.0):    {all_normal:4d} days ({100*all_normal/N:.1f}%)")
print(f"  Slow OU binding:     {slow_binding:4d} days ({100*slow_binding/N:.1f}%)")
print(f"  Medium HAR binding:  {medium_binding:4d} days ({100*medium_binding/N:.1f}%)")
print(f"  Fast IV binding:     {fast_binding:4d} days ({100*fast_binding/N:.1f}%)")

# ------------------------------------------------------------
# 5) Plots
# ------------------------------------------------------------
wealth_df = pd.DataFrame({
    "PC1_base": (1.0 + series["PC1_base"]).cumprod(),
    "Slow_OU_selected_net_CS": (1.0 + series["Slow_OU_selected_net_CS"]).cumprod(),
    "Combined_selectedOU_net_CS": (1.0 + series["Combined_selectedOU_net_CS"]).cumprod(),
    "QQQ": (1.0 + series["QQQ"]).cumprod(),
    "EW_universe": (1.0 + series["EW_universe_gross"]).cumprod(),
}, index=benchmark_idx).dropna()

exp_df = pd.DataFrame({
    "slow_OU_selected": exp_slow,
    "medium_HAR": exp_medium,
    "fast_IV": exp_fast,
    "combined": exp_combined,
}, index=common_idx)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
wealth_df.plot(ax=axes[0], title="Equity curves — base, selected OU overlay, combined overlay, benchmarks")
axes[0].set_ylabel("Cumulative wealth")
exp_df.plot(ax=axes[1], title="Applied exposure by layer", alpha=0.75)
axes[1].axhline(1.0, lw=0.8, ls="--")
axes[1].set_ylabel("Exposure")
dd_df = pd.DataFrame({
    name: (1.0 + r).cumprod() / (1.0 + r).cumprod().cummax() - 1.0
    for name, r in {
        "PC1_base": series["PC1_base"],
        "Slow_OU_selected_net_CS": series["Slow_OU_selected_net_CS"],
        "Combined_selectedOU_net_CS": series["Combined_selectedOU_net_CS"],
        "QQQ": series["QQQ"],
    }.items()
}, index=benchmark_idx).dropna()
dd_df.plot(ax=axes[2], title="Drawdown — base vs retained overlay vs combined vs QQQ")
axes[2].set_ylabel("Drawdown")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6) Bootstrap OOS uncertainty
# ------------------------------------------------------------
boot_slow_summary, ds_slow, dm_slow = bootstrap_compare(
    series["PC1_base"], series["Slow_OU_selected_net_CS"], "Slow_OU_selected_net_CS - base",
    n_boot=BOOT_B_FINAL, block_len=BLOCK_LEN, seed=42
)
boot_comb_summary, ds_comb, dm_comb = bootstrap_compare(
    series["PC1_base"], series["Combined_selectedOU_net_CS"], "Combined_selectedOU_net_CS - base",
    n_boot=BOOT_B_FINAL, block_len=BLOCK_LEN, seed=43
)
boot_summary = pd.concat([boot_slow_summary, boot_comb_summary])

print(f"\nBootstrap OOS uncertainty (N={BOOT_B_FINAL}, mean block={BLOCK_LEN}d, method={BLOCK_METHOD}):")
display(boot_summary.round(4))

# ------------------------------------------------------------
# 7) Save and audit
# ------------------------------------------------------------
combined_overlay = {
    "series": series,
    "perf_table": results,
    "cost_diag_layers": cost_diag_layers,
    "boot_summary": boot_summary,
    "exp_slow": exp_slow,
    "exp_medium": exp_medium,
    "exp_fast": exp_fast,
    "exp_combined": exp_combined,
    "tc_slow": tc_slow,
    "tc_medium": tc_medium,
    "tc_fast": tc_fast,
    "tc_combined": tc_combined,
    "portfolio_oneway_cost": portfolio_oneway_cost,
    "bootstrap_block_len": BLOCK_LEN,
    "bootstrap_block_method": BLOCK_METHOD,
    "cost_source": COST_SOURCE,
}

required_rows = [
    "PC1_base",
    "Slow_OU_selected_gross", "Slow_OU_selected_net_CS",
    "Medium_HAR_gross", "Medium_HAR_net_CS",
    "Fast_IV_gross", "Fast_IV_net_CS",
    "Combined_selectedOU_gross", "Combined_selectedOU_net_CS",
    "QQQ", "EW_universe_gross",
]
for row in required_rows:
    assert row in results.index, f"Missing final performance row: {row}"

assert series["Slow_OU_selected_gross"].index.equals(series["Slow_OU_selected_net_CS"].index)
assert series["Combined_selectedOU_gross"].index.equals(series["Combined_selectedOU_net_CS"].index)
assert cost_diag_layers.loc["Slow_OU_selected", "total_overlay_cost_%"] >= 0
assert cost_diag_layers.loc["Combined_selectedOU", "total_overlay_cost_%"] >= 0

print("\nSaved: combined_overlay")
print("Done — final selected-OU overlay framework complete.")


## 9) Notes and next steps

Current retained result:
- the **DEV-AUC-selected OU log-volatility exceedance overlay** is the strongest net OOS overlay;
- the combined architecture is useful as a multi-layer design, but does not dominate the selected OU layer net of costs;
- HAR-RV remains a useful medium-speed benchmark/component;
- the fast IV layer is an implemented challenger and should eventually be replaced/tested against a Kalman innovation / Mahalanobis surprise detector.

Important limitation:
- the OU design sweep uses development-sample AUC because development-period overlay P&L is degenerate: the development window is short/calm and `dev_trades=0` for all candidates. This must be disclosed in README/paper writing.
